# 06 — Folium launch sites

Marqueurs géographiques, MarkerCluster, distances au littoral (calcul indicatif à partir des coordonnées existantes).

Source : jeu historique IBM DS0321EN. Exécuter toutes les cellules dans Google Colab. Les sorties sont produites à l’exécution, pas préremplies.

In [ ]:
%pip -q install folium
import pandas as pd, folium
from folium.plugins import MarkerCluster, MousePosition, MiniMap
from IPython.display import display
from pathlib import Path
BASE = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/"
df = pd.read_csv(BASE + "dataset_part_2.csv")
required={'LaunchSite','Latitude','Longitude','Class'}
assert required.issubset(df.columns), f'Missing columns: {required-set(df.columns)}'
OUT=Path('outputs'); OUT.mkdir(exist_ok=True)
sites=df.groupby(['LaunchSite','Latitude','Longitude'],dropna=False).agg(launches=('Class','size'),success_rate=('Class','mean')).reset_index()
display(sites)

In [ ]:
center=[df['Latitude'].mean(),df['Longitude'].mean()]
m=folium.Map(location=center,zoom_start=4,tiles='OpenStreetMap')
for _,s in sites.iterrows():
    folium.Circle(location=[s.Latitude,s.Longitude],radius=8000,color='blue',fill=True,fill_opacity=.12,
      popup=f"{s.LaunchSite}: {s.launches} launches / {s.success_rate:.1%} landing success").add_to(m)
    folium.Marker([s.Latitude,s.Longitude],tooltip=str(s.LaunchSite)).add_to(m)
m.add_child(MousePosition()); m.add_child(MiniMap())
m.save(str(OUT/'launch_sites_map.html')); display(m)

In [ ]:
m2=folium.Map(location=center,zoom_start=4)
cluster=MarkerCluster(name='Launch results').add_to(m2)
for _,r in df.dropna(subset=['Latitude','Longitude']).iterrows():
    success=int(r['Class'])
    folium.Marker(location=[r['Latitude'],r['Longitude']],
      popup=f"Flight {r['FlightNumber']} | {r['LaunchSite']} | {'Success' if success else 'Failure'}",
      icon=folium.Icon(color='green' if success else 'red',icon='info-sign')).add_to(cluster)
folium.LayerControl().add_to(m2); m2.save(str(OUT/'launch_results_cluster.html')); display(m2)
print('Saved interactive HTML maps. Take screenshots of the rendered maps for the PDF.')

In [ ]:
from google.colab import files
files.download(str(OUT/'launch_sites_map.html'))
files.download(str(OUT/'launch_results_cluster.html'))